In [1]:
import os
import pandas as pd


In [ ]:


def analyze_csv(file_path):
    """
    Analyze a single CSV file to extract various statistics.
    
    Parameters:
    file_path (str): Path to the CSV file.
    
    Returns:
    dict: A dictionary containing the computed statistics.
    """
    df = pd.read_csv(file_path)
    
    # Ensure required columns exist
    required_columns = ["frame.time", "frame.len", "ip.src", "ip.dst", "ip.proto"] #"eth.src", "eth.dst", in visQuic
    for col in required_columns:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")
        
    # Convert necessary columns to numeric (handling errors gracefully)
    df["frame.time"] = pd.to_numeric(df["frame.time"], errors="coerce")
    df["frame.len"] = pd.to_numeric(df["frame.len"], errors="coerce")
    df["ip.proto"] = pd.to_numeric(df["ip.proto"], errors="coerce")

    stats = {}

    # Calculate the average recording time (difference between min and max frame time)
    stats["avg_recording_time"] = df["frame.time"].max()

    # Calculate average packet size
    stats["avg_packet_size"] = df["frame.len"].mean()

    # Count the number of packets
    stats["packet_count"] = len(df)

    # Count unique IP addresses (src and dst)
    stats["unique_ips"] = len(pd.concat([df["ip.src"], df["ip.dst"]]).unique())

    # Count unique MAC addresses (src and dst)
    # stats["unique_macs"] = len(pd.concat([df["eth.src"], df["eth.dst"]]).unique())

    # Calculate direction distribution (0 for client to server, 1 for server to client)
    df["Direction"] = df["ip.src"].apply(lambda x: 0 if x == "0.0.0.0" else 1)
    stats["client_to_server"] = df[df["Direction"] == 0].shape[0]
    stats["server_to_client"] = df[df["Direction"] == 1].shape[0]

    # Protocol distribution (count occurrences of unique protocols)
    stats["protocol_distribution"] = df["ip.proto"].value_counts().to_dict()

    # Return the computed statistics
    return stats

def explore_directory(input_dir):
    """
    Recursively explore all CSV files in a directory and its subdirectories,
    extracting various statistics for each file and each class.
    
    Parameters:
    input_dir (str): Path to the directory containing the CSV files.
    
    Returns:
    dict: A dictionary with statistics for each class.
    """
    analysis_results = {}

    # Get the top-level directories (i.e., the class directories)
    class_directories = [d for d in os.listdir(input_dir) if os.path.isdir(os.path.join(input_dir, d))]

    # Iterate through each class directory (Youtube, Semrush, Discord)
    for class_name in class_directories:
        # Initialize a dictionary to hold stats for this class
        class_dir = os.path.join(input_dir, class_name)
        analysis_results[class_name] = {
            "total_recording_time": 0,
            "total_packet_count": 0,
            "longest_recording_time":-1,
            "biggest_packet_count":-1,
            "total_unique_ips": set(),
            # "total_unique_macs": set(),
            "protocol_distribution": {},
            "client_to_server": 0,
            "server_to_client": 0,
            "file_count": 0
        }

        # Walk through the subdirectories within the current class directory
        for root, dirs, files in os.walk(class_dir):
            # Skip non-CSV files
            csv_files = [f for f in files if f.endswith('.csv')]
            
            if not csv_files:
                continue  # Skip if there are no CSV files in this directory
            
            # Process each CSV file in the current directory
            for file in csv_files:
                file_path = os.path.join(root, file)
                stats = analyze_csv(file_path)
                
                # Update the overall statistics for the class
                analysis_results[class_name]["longest_recording_time"] = max(analysis_results[class_name]["longest_recording_time"],
                                                                             stats["avg_recording_time"])
                analysis_results[class_name]["total_recording_time"] += stats["avg_recording_time"]
                analysis_results[class_name]["biggest_packet_count"] = max(analysis_results[class_name]["biggest_packet_count"],
                                                                           stats["packet_count"])
                analysis_results[class_name]["total_packet_count"] += stats["packet_count"]
                analysis_results[class_name]["total_unique_ips"].update(pd.read_csv(file_path)[["ip.src", "ip.dst"]].stack().unique())
                # analysis_results[class_name]["total_unique_macs"].update(pd.read_csv(file_path)[["eth.src", "eth.dst"]].stack().unique())
                analysis_results[class_name]["protocol_distribution"].update(stats["protocol_distribution"])
                analysis_results[class_name]["client_to_server"] += stats["client_to_server"]
                analysis_results[class_name]["server_to_client"] += stats["server_to_client"]
                analysis_results[class_name]["file_count"] += 1

    # Calculate averages for each class
    for class_name, stats in analysis_results.items():
        stats["avg_recording_time"] = stats["total_recording_time"] / stats["file_count"] if stats["file_count"] > 0 else 0
        stats["avg_packet_size"] = stats["total_packet_count"] / stats["file_count"] if stats["file_count"] > 0 else 0
        stats["unique_ips_count"] = len(stats["total_unique_ips"])
        # stats["unique_macs_count"] = len(stats["total_unique_macs"])
        stats["avg_client_to_server"] = stats["client_to_server"] / stats["file_count"] if stats["file_count"] > 0 else 0
        stats["avg_server_to_client"] = stats["server_to_client"] / stats["file_count"] if stats["file_count"] > 0 else 0

    return analysis_results

def print_class_statistics(statistics):
    """
    Print the statistics for each class in a readable format.
    
    Parameters:
    statistics (dict): A dictionary containing statistics for each class.
    """
    for class_name, stats in statistics.items():
        print(f"\nClass: {class_name}")
        print(f"Total Files Processed: {stats['file_count']}")
        print(f"Average Recording Time: {stats['avg_recording_time']:.2f} seconds")
        print(f"Average Packets size: {stats['avg_packet_size']}")
        print(f"Longest recording time: {stats['longest_recording_time']:.2f} seconds")
        print(f"Biggest Packets size: {stats['biggest_packet_count']}")
        print(f"Unique IPs: {stats['unique_ips_count']}")
        # print(f"Unique MACs: {stats['unique_macs_count']}")
        print(f"Traffic Direction (Client to Server): {stats['avg_client_to_server']} packets")
        print(f"Traffic Direction (Server to Client): {stats['avg_server_to_client']} packets")
        print(f"Protocol Distribution: {stats['protocol_distribution']}")


In [5]:

# Example Usage
input_dir = '../../../data/Chanan/UTMobileNet/Deterministic Automated Data'  # Replace with the path to your data directory
statistics = explore_directory(input_dir)
print_class_statistics(statistics)


/tmp/ipykernel_1096008/3393552472.py:11: DtypeWarning: Columns (39) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
/tmp/ipykernel_1096008/3393552472.py:102: DtypeWarning: Columns (39) have mixed types. Specify dtype option on import or set low_memory=False.
  analysis_results[class_name]["total_unique_ips"].update(pd.read_csv(file_path)[["ip.src", "ip.dst"]].stack().unique())
/tmp/ipykernel_1096008/3393552472.py:11: DtypeWarning: Columns (33,39) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
/tmp/ipykernel_1096008/3393552472.py:102: DtypeWarning: Columns (33,39) have mixed types. Specify dtype option on import or set low_memory=False.
  analysis_results[class_name]["total_unique_ips"].update(pd.read_csv(file_path)[["ip.src", "ip.dst"]].stack().unique())
/tmp/ipykernel_1096008/3393552472.py:11: DtypeWarning: Columns (39) have mixed types. Specify dtype option on import or se


Class: youtube
Total Files Processed: 237
Average Recording Time: nan seconds
Average Packets size: 900.5991561181435
Longest recording time: -1.00 seconds
Biggest Packets size: 8992
Unique IPs: 163
Traffic Direction (Client to Server): 0.0 packets
Traffic Direction (Server to Client): 900.5991561181435 packets
Protocol Distribution: {17: 1088, 6: 114, 2: 4, '17': 1082, '6': 106, '2': 4, '1,17': 2}

Class: dropbox
Total Files Processed: 245
Average Recording Time: nan seconds
Average Packets size: 752.6408163265306
Longest recording time: -1.00 seconds
Biggest Packets size: 1657
Unique IPs: 295
Traffic Direction (Client to Server): 0.0 packets
Traffic Direction (Server to Client): 752.6408163265306 packets
Protocol Distribution: {6: 520, 17: 14, '6': 362, '17': 12, '1,17': 2, 2.0: 2, '2': 2}

Class: facebook
Total Files Processed: 267
Average Recording Time: nan seconds
Average Packets size: 2903.423220973783
Longest recording time: -1.00 seconds
Biggest Packets size: 20737
Unique IPs

# fiding Nan problem

In [ ]:
import os
import pandas as pd

def analyze_csv(file_path):
    """Analyzes a single CSV file and extracts relevant statistics."""
    try:
        df = pd.read_csv(file_path)

        # Ensure the required column exists
        if "frame.time" not in df.columns:
            print(f"⚠️ Warning: 'frame.time' column missing in {file_path}")
            return None

        # Ensure there are valid (non-NaN) values in the column
        if df["frame.time"].isna().all():
            print(f"⚠️ Warning: All values in 'frame.time' are NaN in {file_path}")
            return None

        if df.empty:
            print(f"⚠️ Warning: CSV file {file_path} is empty.")
            return None

        min_time = df["frame.time"].min()
        max_time = df["frame.time"].max()

        if pd.isna(min_time) or pd.isna(max_time):
            print(f"⚠️ Warning: min or max 'frame.time' is NaN in {file_path}")
            return None

        duration = max_time - min_time
        print(f"✅ {file_path} - Min: {min_time}, Max: {max_time}, Duration: {duration}")

        return {
            "avg_recording_time": duration,
            "packet_count": len(df)
        }

    except Exception as e:
        print(f"🚨 Error processing {file_path}: {e}")
        return None

def explore_directory(base_dir):
    """Recursively explores a directory, analyzes CSV files, and groups data by class (Youtube, Discord, Semrush)."""
    analysis_results = {}

    for class_name in os.listdir(base_dir):  # Iterate over class directories
        class_path = os.path.join(base_dir, class_name)

        if not os.path.isdir(class_path):
            continue  # Skip non-directory files

        print(f"\n🔎 Processing Class: {class_name}")

        # Initialize statistics for this class
        analysis_results[class_name] = {
            "total_recording_time": 0,
            "total_packet_count": 0,
            "valid_files": 0
        }

        # Walk through subdirectories and files
        for root, _, files in os.walk(class_path):
            for file in files:
                if file.endswith(".csv"):
                    file_path = os.path.join(root, file)

                    stats = analyze_csv(file_path)
                    if stats is None:
                        print(f"🚨 Skipping {file_path} due to invalid data.")
                        continue  # Skip problematic files

                    # Aggregate stats
                    analysis_results[class_name]["total_recording_time"] += stats["avg_recording_time"]
                    analysis_results[class_name]["total_packet_count"] += stats["packet_count"]
                    analysis_results[class_name]["valid_files"] += 1

        # Compute average recording time per file
        if analysis_results[class_name]["valid_files"] > 0:
            analysis_results[class_name]["average_recording_time"] = (
                analysis_results[class_name]["total_recording_time"] / analysis_results[class_name]["valid_files"]
            )
        else:
            analysis_results[class_name]["average_recording_time"] = None  # No valid files found

    return analysis_results


In [2]:

# Example Usage
input_dir = '/home/chanan/Time-Series-Library/VisQUIC/csv'  # Replace with the path to your data directory
statistics = explore_directory(input_dir)
# print_class_statistics(statistics)



🔎 Processing Class: youtube.com
✅ /home/chanan/Time-Series-Library/VisQUIC/csv/youtube.com/https___www.youtube.com_intl_ALL_se_ads_/https___www.youtube.com_intl_ALL_se_ads_-00330_quic_anonymized_filtered.csv - Min: 0.0, Max: 2.053716, Duration: 2.053716
✅ /home/chanan/Time-Series-Library/VisQUIC/csv/youtube.com/https___www.youtube.com_intl_ALL_se_ads_/https___www.youtube.com_intl_ALL_se_ads_-00250_quic_anonymized_filtered.csv - Min: 0.0, Max: 1.438771, Duration: 1.438771
✅ /home/chanan/Time-Series-Library/VisQUIC/csv/youtube.com/https___www.youtube.com_intl_ALL_se_ads_/https___www.youtube.com_intl_ALL_se_ads_-00490_quic_anonymized_filtered.csv - Min: 0.0, Max: 1.878088, Duration: 1.878088
✅ /home/chanan/Time-Series-Library/VisQUIC/csv/youtube.com/https___www.youtube.com_intl_ALL_se_ads_/https___www.youtube.com_intl_ALL_se_ads_-00130_quic_anonymized_filtered.csv - Min: 0.0, Max: 1.762918, Duration: 1.762918
✅ /home/chanan/Time-Series-Library/VisQUIC/csv/youtube.com/https___www.youtube.